<a href="https://colab.research.google.com/github/KuChoiLab/2026_bomun_workshop/blob/main/WGS_practice/WGS_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Whole Genome Sequencing analysis Workflow

## 1. 분석에 필요한 환경설정

** Reference genome 때문에 파일 용량이 매우 큼 -> 해당 부분 수정 예정


In [1]:
# 2분 30초 정도 소요

import gdown

file_id = "1Nl9fXbUDOHNANonYRy2_LLKDe5cJuotI"
output = "/content/WGS.tar" # 저장 위치 및 저장할 파일 이름
gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1Nl9fXbUDOHNANonYRy2_LLKDe5cJuotI
From (redirected): https://drive.google.com/uc?id=1Nl9fXbUDOHNANonYRy2_LLKDe5cJuotI&confirm=t&uuid=b1e62b91-bc07-400e-ad8a-0d800c93dbf4
To: /content/WGS.tar
100%|██████████| 13.3G/13.3G [02:32<00:00, 87.5MB/s]


'/content/WGS.tar'

In [2]:
# 2분정도 소요

!tar -xf /content/WGS.tar

In [28]:
import os
os.environ['PATH'] = "/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/conda/bin:" + os.environ['PATH']

In [29]:
!which conda
!conda --version

/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/conda/bin/conda
/bin/bash: /content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/conda/bin/conda: /content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/conda/bin/python: bad interpreter: No such file or directory


In [4]:
# samtools dependency error -> apt-get install 필요

!apt-get install libncurses5

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libtinfo5
The following NEW packages will be installed:
  libncurses5 libtinfo5
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 207 kB of archives.
After this operation, 883 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libtinfo5 amd64 6.3-2ubuntu0.1 [100 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libncurses5 amd64 6.3-2ubuntu0.1 [107 kB]
Fetched 207 kB in 1s (265 kB/s)
Selecting previously unselected package libtinfo5:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libtinfo5_6.3-2ubuntu0.1_amd64.deb ...
Unpacking libtinfo5:amd64 (6.3-2ubuntu0.1) ...
Selecting previously unselected package libncurses5:amd64.
Preparing to unpack .../libncurses5_6.3-2ubuntu0.1_amd64

## 2. Alignment

In [17]:
%cd /content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/Raw_data

/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/Raw_data


In [19]:
#make sample list

%%bash
ls -d * > input_path.txt

In [20]:
!head input_path.txt

10_ji0suk
13_gim0gi
17_jang0jae
21_gim0rae
22_jang0ji
23_jeong0gi
25_bak0gyeong
26_gim0hyeon
27_gim0gyeong
28_yun0hoe


In [32]:
# run bwa mem

%%bash

while IFS= read -r line; do

sample_id=$line

mkdir -p ${sample_id}/alignment

bwa mem -t 2 -Y \
  -R "@RG\tID:${sample_id}\tPL:ILLUMINA\tPU:${sample_id}\tSM:${sample_id}\tLB:${sample_id}" \
  "/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/Reference/hs38DH.fasta" \
  "${sample_id}/obesity_fastq/${sample_id}_R1_001.fastq.gz" \
  "${sample_id}/obesity_fastq/${sample_id}_R2_001.fastq.gz" \
  | samtools sort -m 2G -o ${sample_id}/alignment/${sample_id}.bam -O bam
done < input_path.txt

Process is interrupted.


In [ ]:
# Output : Bam file

!samtools view ./10_ji0suk/alignment/10_ji0suk.bam | head -1

## 3. MarkDuplicate

In [ ]:
%%bash

export gatk=/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/gatk-4.3.0.0/gatk-package-4.3.0.0-local.jar

while IFS= read -r line; do
  sample_id=$line

  mkdir -p ${sample_id}/markdup

  java -jar $gatk -I=${sample_id}/alignment/${sample_id}.bam \
  -O=${sample_id}/markdup/dedup.${sample_id}.bam \
  -M=${sample_id}/markdup/markdups_${sample_id}.txt \
  ASSUME_SORT_ORDER=coordinate \
  MAX_RECORDS_IN_RAM=2000000 \
  COMPRESSION_LEVEL=1 \
  CREATE_INDEX=true \
  VALIDATION_STRINGENCY=SILENT

done < input_path.txt


## 4. BQSR (applyBQSR만 진행)

In [ ]:
%%bash
export gatk=/content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/gatk-4.3.0.0/gatk-package-4.3.0.0-local.jar


while IFS= read -r line; do
  sample_id=$line

  mkdir -p ${sample_id}/BQSR/

  java -jar $gatk -R /content/content/drive/MyDrive/bomun_hands_on_exercise/WGS_practice/Reference/hs38DH.fasta \
  -I ${sample_id}/markdup/dedup.${sample_id}.bam -O ${sample_id}/BQSR/${sample_id}.cram --bqsr ${sample_id}/${sample_id}.recal.table

  samtools index ${sample_id}/BQSR/${sample_id}.cram


done < input_path.txt

## 5. Variant Calling (Haplotypecaller)